# Dataset Report — `tomato_leaf_disease_v1`

Notebook chỉ **đọc** (không build lại) Kaggle Dataset đã được `01_build_tomato_leaf_disease_v1.ipynb` sinh ra, để xác nhận nhanh bộ dữ liệu cuối cùng trước khi dùng cho training.

## Trước khi chạy
1. **Add Input** → chọn Kaggle Dataset output của `01_build_tomato_leaf_disease_v1.ipynb` (dataset đã bấm **Save Version → Save & Run All (Commit)** thành công).
2. Không cần Internet, không cần GPU/Accelerator.

## Notebook này làm gì
1. Tự tìm `data.yaml` khớp đúng 4 class của `tomato_leaf_disease_v1` trong `/kaggle/input` (không đoán đường dẫn).
2. Thống kê số ảnh / số box theo class + số ảnh **negative** (lá khỏe, label rỗng) cho từng split (`train`, `val`, `test`) + dung lượng đĩa — tính trực tiếp từ file thật.
3. Thống kê kích thước ảnh (min/max/mean width-height, lấy mẫu ngẫu nhiên).
4. Vẽ biểu đồ phân bố class + negative theo split.
5. Hiển thị ảnh ví dụ kèm bounding box cho ảnh dương, và ảnh ví dụ negative (lá khỏe) riêng để xác nhận không sót box.
6. In lại `dataset_report.md` / `licenses.md` / `zenodo_audit.md` gốc (nếu có trong input) để có báo cáo đầy đủ ở một chỗ.

> Lưu ý: notebook này vẽ nhiều ảnh mẫu (`plt.show()`) nên file `.ipynb` sau khi chạy sẽ lớn hơn do nhúng ảnh base64 vào output — nếu import qua GitHub bị lỗi "Failed to fetch this content", xóa output các cell trước khi push (đã từng gặp ở `01_build_tomato_ripeness_v1.ipynb` do vượt giới hạn ~1MB của GitHub Contents API).

In [ ]:
import random
from collections import defaultdict
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

random.seed(42)
np.random.seed(42)

TARGET_CLASSES = ["leaf_early_blight", "leaf_late_blight", "leaf_mold", "leaf_septoria_spot"]
SPLITS = ["train", "val", "test"]


def find_dataset_root():
    """Tìm thư mục chứa data.yaml của tomato_leaf_disease_v1 (khớp đúng 4 class, không đoán đường dẫn)."""
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


def find_manifest_root():
    """Tìm thư mục manifests (chứa class_distribution.csv) sinh ra bởi 01_build_tomato_leaf_disease_v1.ipynb."""
    hits = list(Path("/kaggle/input").rglob("class_distribution.csv"))
    return hits[0].parent if hits else None


DATASET_ROOT = find_dataset_root()
MANIFEST_ROOT = find_manifest_root()

if DATASET_ROOT is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 4 class của tomato_leaf_disease_v1 trong /kaggle/input.\n"
        "Kiểm tra đã Add Input đúng Kaggle Dataset xuất ra từ 01_build_tomato_leaf_disease_v1.ipynb "
        "(SAU KHI notebook đó đã Save Version) chưa."
    )

print("DATASET_ROOT :", DATASET_ROOT)
print("MANIFEST_ROOT:", MANIFEST_ROOT if MANIFEST_ROOT else "(không tìm thấy -> phần báo cáo gốc ở cuối sẽ bị bỏ qua)")

In [ ]:
data_yaml = yaml.safe_load((DATASET_ROOT / "data.yaml").read_text(encoding="utf-8"))
print(yaml.safe_dump(data_yaml, allow_unicode=True, sort_keys=False))

for split in SPLITS:
    split_dir = DATASET_ROOT / split
    print(f"{split:10s} tồn tại: {split_dir.exists()}  ->  {split_dir}")

## Thống kê tổng quan
Số ảnh, số box theo class, số ảnh negative (label rỗng), và dung lượng đĩa cho từng split — tính trực tiếp từ file thật trong dataset (không phụ thuộc CSV cũ đã sinh sẵn).

In [ ]:
def dir_size_mb(path):
    total = sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file())
    return total / (1024 * 1024)


def split_stats(split):
    split_dir = DATASET_ROOT / split
    images_dir, labels_dir = split_dir / "images", split_dir / "labels"
    if not images_dir.exists():
        return None
    n_images = len(list(images_dir.glob("*")))
    class_counts = defaultdict(int)
    n_negative = 0
    n_boxes = 0
    for txt in labels_dir.glob("*.txt"):
        lines = [l for l in txt.read_text().splitlines() if l.strip()]
        if not lines:
            n_negative += 1
            continue
        for line in lines:
            cid = int(line.split()[0])
            class_counts[TARGET_CLASSES[cid]] += 1
            n_boxes += 1
    sources = sorted({p.name.split("__", 1)[0] for p in images_dir.glob("*") if "__" in p.name})
    return {
        "split": split,
        "n_images": n_images,
        "n_negative": n_negative,
        "n_boxes": n_boxes,
        "sources": ", ".join(sources),
        "disk_mb": round(dir_size_mb(split_dir), 1),
        **{c: class_counts.get(c, 0) for c in TARGET_CLASSES},
    }


stats_rows = [s for s in (split_stats(sp) for sp in SPLITS) if s is not None]
overview_df = pd.DataFrame(stats_rows)
print(overview_df.to_string(index=False))

print(f"\nTổng ảnh (train+val+test): {overview_df['n_images'].sum()}")
print(f"Tổng ảnh negative (lá khỏe): {overview_df['n_negative'].sum()} "
      f"({100 * overview_df['n_negative'].sum() / overview_df['n_images'].sum():.1f}%)")
print(f"Tổng dung lượng trên đĩa: {overview_df['disk_mb'].sum():.1f} MB")

### Kích thước ảnh
Lấy mẫu ngẫu nhiên tối đa 200 ảnh/split để thống kê min/max/mean width-height.

In [ ]:
def resolution_stats(split, sample_n=200):
    images_dir = DATASET_ROOT / split / "images"
    files = sorted(images_dir.glob("*"))
    sample = random.Random(42).sample(files, min(sample_n, len(files))) if files else []
    sizes = []
    for p in sample:
        try:
            with Image.open(p) as im:
                sizes.append(im.size)
        except Exception:
            pass
    if not sizes:
        return None
    ws, hs = zip(*sizes)
    return {
        "split": split,
        "n_sampled": len(sizes),
        "width_min": min(ws), "width_max": max(ws), "width_mean": round(float(np.mean(ws)), 0),
        "height_min": min(hs), "height_max": max(hs), "height_mean": round(float(np.mean(hs)), 0),
    }


res_rows = [r for r in (resolution_stats(sp) for sp in SPLITS) if r is not None]
res_df = pd.DataFrame(res_rows)
print(res_df.to_string(index=False))

## Phân bố class + negative theo split (biểu đồ)

In [ ]:
plot_df = overview_df.set_index("split")[TARGET_CLASSES + ["n_negative"]]
plot_df = plot_df.rename(columns={"n_negative": "negative (lá khỏe)"})
ax = plot_df.plot(kind="bar", figsize=(9, 5), color=["lime", "orange", "red", "magenta", "#90a4ae"])
ax.set_ylabel("Số lượng bounding box / ảnh negative")
ax.set_title("Phân bố class + negative theo split — tomato_leaf_disease_v1")
ax.legend(title="Class")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/kaggle/working/class_distribution_chart.png", dpi=100, bbox_inches="tight")
plt.show()
plt.close()

## Ảnh ví dụ (kèm bounding box)
Màu box khớp với `01_build_tomato_leaf_disease_v1.ipynb`: xanh lá (lime) = `leaf_early_blight`, cam = `leaf_late_blight`, đỏ = `leaf_mold`, hồng magenta = `leaf_septoria_spot`. Ảnh negative không có box — cần xác nhận đó thực sự là lá khỏe.

In [ ]:
def show_samples(split, only="any", n=6, title=""):
    """only: 'any' | 'positive' (có ít nhất 1 box) | 'negative' (label rỗng)."""
    images_dir = DATASET_ROOT / split / "images"
    labels_dir = DATASET_ROOT / split / "labels"
    all_imgs = sorted(images_dir.glob("*"))

    def label_lines(img_path):
        lp = labels_dir / (img_path.stem + ".txt")
        if not lp.exists():
            return []
        return [l for l in lp.read_text().splitlines() if l.strip()]

    if only == "positive":
        all_imgs = [p for p in all_imgs if label_lines(p)]
    elif only == "negative":
        all_imgs = [p for p in all_imgs if not label_lines(p)]

    if not all_imgs:
        print(f"[{title}] Không có ảnh để hiển thị (only={only}).")
        return
    sample = random.Random(42).sample(all_imgs, min(n, len(all_imgs)))

    fig, axes = plt.subplots(1, len(sample), figsize=(3.2 * len(sample), 3.2))
    axes = [axes] if len(sample) == 1 else axes
    colors = ["lime", "orange", "red", "magenta"]
    for ax, img_path in zip(axes, sample):
        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)
        for line in label_lines(img_path):
            cid, xc, yc, bw, bh = line.split()
            cid = int(cid)
            xc, yc, bw, bh = map(float, (xc, yc, bw, bh))
            xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
            rect = patches.Rectangle((xmin, ymin), bw * w, bh * h, linewidth=1.5,
                                      edgecolor=colors[cid], facecolor="none")
            ax.add_patch(rect)
            ax.text(xmin, max(ymin - 4, 0), TARGET_CLASSES[cid], color=colors[cid], fontsize=7, weight="bold")
        ax.set_title(img_path.name, fontsize=6)
        ax.axis("off")
    fig.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/sample_{title}.png", dpi=80, bbox_inches="tight")
    plt.show()
    plt.close(fig)


show_samples("train", only="positive", n=6, title="train_positive_boxes")
show_samples("train", only="negative", n=6, title="train_negative_healthy")
show_samples("val", only="positive", n=6, title="val_positive_boxes")
show_samples("test", only="positive", n=6, title="test_positive_boxes")

## Báo cáo & manifest gốc (từ `01_build_tomato_leaf_disease_v1.ipynb`)
In lại nguyên văn nếu thư mục `manifests/` có trong input, để có đủ thông tin ở một chỗ (không cần mở lại notebook build). Bao gồm cả `zenodo_audit.md` — nhắc lại rằng `leaf_zenodo` **chưa được gộp** vào dataset này.

In [ ]:
if MANIFEST_ROOT:
    for fname in ["dataset_report.md", "licenses.md", "zenodo_audit.md"]:
        fpath = MANIFEST_ROOT / fname
        if fpath.exists():
            print(f"===== {fname} =====")
            print(fpath.read_text(encoding="utf-8"))
            print()

    rejected_csv = MANIFEST_ROOT / "rejected_images.csv"
    dup_csv = MANIFEST_ROOT / "duplicate_report.csv"
    if rejected_csv.exists():
        print(f"rejected_images.csv: {len(pd.read_csv(rejected_csv))} dòng "
              "(ảnh chỉ có bệnh ngoài phạm vi MVP, không phải Healthy -> bị loại hoàn toàn)")
    if dup_csv.exists():
        print(f"duplicate_report.csv: {len(pd.read_csv(dup_csv))} dòng "
              "(ảnh nằm trong nhóm trùng/near-duplicate, kể cả bản augmented cùng gốc)")
else:
    print("Không tìm thấy thư mục manifests/ trong input -> bỏ qua phần báo cáo gốc.")

## Kết luận
Nếu số liệu ở trên hợp lý (không split nào `n_images = 0`, phân bố class không lệch bất thường, ảnh mẫu dương gắn đúng nhãn, ảnh mẫu negative thực sự là lá khỏe) thì `tomato_leaf_disease_v1` đã sẵn sàng làm input cho **`03_train_leaf_baseline.ipynb`**.